In [1]:
import pandas as pd
import numpy as np

from pymongo import MongoClient
from dotenv import load_dotenv
import os

from sklearn.model_selection import train_test_split

In [2]:
load_dotenv()

MONGODB_URI = os.getenv("MONGODB_URI")

if not MONGODB_URI:
    raise RuntimeError("MONGODB_URI not found.")

In [3]:
client = MongoClient(MONGODB_URI)

db = client["aqi_predictor"]

collection = db["aqi_features"]

df = pd.DataFrame(list(collection.find()))

df.head()

,_id,timestamp,city,aqi,aqi_change_rate,co,day,day_of_week,hour,humidity,month,no2,o3,pm10,pm25,pressure,so2,temperature,wind_speed
0,6a8022d8a2c1b4faf9cbb500,2025-08-15T09:00:00+00:00,karachi,58,0.0,80.85,15,4,9,58,8,0.05,39.96,52.39,15.65,999.7,0.15,32.7,11.5
1,6a8022d8a2c1b4faf9cbb502,2025-08-15T10:00:00+00:00,karachi,59,1.0,80.90,15,4,10,63,8,0.05,39.21,53.08,16.09,998.7,0.15,31.9,13.4
2,6a8022d8a2c1b4faf9cbb503,2025-08-15T11:00:00+00:00,karachi,60,1.0,80.73,15,4,11,63,8,0.05,38.67,54.27,16.60,998.1,0.15,31.5,13.2
3,6a8022d8a2c1b4faf9cbb504,2025-08-15T12:00:00+00:00,karachi,62,2.0,80.36,15,4,12,72,8,0.06,38.42,55.99,17.14,998.5,0.16,30.3,12.7
4,6a8022d8a2c1b4faf9cbb505,2025-08-15T13:00:00+00:00,karachi,63,1.0,80.13,15,4,13,71,8,0.08,38.36,58.14,17.65,999.1,0.16,30.2,9.7


In [4]:
if "_id" in df.columns:
    df.drop(columns="_id", inplace=True)

df.head()

,timestamp,city,aqi,aqi_change_rate,co,day,day_of_week,hour,humidity,month,no2,o3,pm10,pm25,pressure,so2,temperature,wind_speed
0,2025-08-15T09:00:00+00:00,karachi,58,0.0,80.85,15,4,9,58,8,0.05,39.96,52.39,15.65,999.7,0.15,32.7,11.5
1,2025-08-15T10:00:00+00:00,karachi,59,1.0,80.90,15,4,10,63,8,0.05,39.21,53.08,16.09,998.7,0.15,31.9,13.4
2,2025-08-15T11:00:00+00:00,karachi,60,1.0,80.73,15,4,11,63,8,0.05,38.67,54.27,16.60,998.1,0.15,31.5,13.2
3,2025-08-15T12:00:00+00:00,karachi,62,2.0,80.36,15,4,12,72,8,0.06,38.42,55.99,17.14,998.5,0.16,30.3,12.7
4,2025-08-15T13:00:00+00:00,karachi,63,1.0,80.13,15,4,13,71,8,0.08,38.36,58.14,17.65,999.1,0.16,30.2,9.7


In [5]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

df = df.sort_values("timestamp")

df.reset_index(drop=True, inplace=True)

df.head()

,timestamp,city,aqi,aqi_change_rate,co,day,day_of_week,hour,humidity,month,no2,o3,pm10,pm25,pressure,so2,temperature,wind_speed
0,2025-08-15 09:00:00+00:00,karachi,58,0.0,80.85,15,4,9,58,8,0.05,39.96,52.39,15.65,999.7,0.15,32.7,11.5
1,2025-08-15 10:00:00+00:00,karachi,59,1.0,80.90,15,4,10,63,8,0.05,39.21,53.08,16.09,998.7,0.15,31.9,13.4
2,2025-08-15 11:00:00+00:00,karachi,60,1.0,80.73,15,4,11,63,8,0.05,38.67,54.27,16.60,998.1,0.15,31.5,13.2
3,2025-08-15 12:00:00+00:00,karachi,62,2.0,80.36,15,4,12,72,8,0.06,38.42,55.99,17.14,998.5,0.16,30.3,12.7
4,2025-08-15 13:00:00+00:00,karachi,63,1.0,80.13,15,4,13,71,8,0.08,38.36,58.14,17.65,999.1,0.16,30.2,9.7


In [6]:
FORECAST_HOURS = 72

df["target_aqi"] = df["aqi"].shift(-FORECAST_HOURS)

df.head()

,timestamp,city,aqi,aqi_change_rate,co,day,day_of_week,hour,humidity,month,no2,o3,pm10,pm25,pressure,so2,temperature,wind_speed,target_aqi
0,2025-08-15 09:00:00+00:00,karachi,58,0.0,80.85,15,4,9,58,8,0.05,39.96,52.39,15.65,999.7,0.15,32.7,11.5,51.0
1,2025-08-15 10:00:00+00:00,karachi,59,1.0,80.90,15,4,10,63,8,0.05,39.21,53.08,16.09,998.7,0.15,31.9,13.4,51.0
2,2025-08-15 11:00:00+00:00,karachi,60,1.0,80.73,15,4,11,63,8,0.05,38.67,54.27,16.60,998.1,0.15,31.5,13.2,51.0
3,2025-08-15 12:00:00+00:00,karachi,62,2.0,80.36,15,4,12,72,8,0.06,38.42,55.99,17.14,998.5,0.16,30.3,12.7,50.0
4,2025-08-15 13:00:00+00:00,karachi,63,1.0,80.13,15,4,13,71,8,0.08,38.36,58.14,17.65,999.1,0.16,30.2,9.7,49.0


In [7]:
df = df.dropna(subset=["target_aqi"])

df.reset_index(drop=True, inplace=True)

df.tail()

,timestamp,city,aqi,aqi_change_rate,co,day,day_of_week,hour,humidity,month,no2,o3,pm10,pm25,pressure,so2,temperature,wind_speed,target_aqi
8419,2026-08-12 04:00:00+00:00,karachi,68,0.0,73.39,12,2,4,75,8,0.10,46.69,82.59,20.39,1002.6,0.41,28.9,13.1,63.0
8420,2026-08-12 05:00:00+00:00,karachi,69,1.0,73.04,12,2,5,72,8,0.09,46.01,87.70,20.79,1002.6,0.41,29.6,13.9,64.0
8421,2026-08-12 06:00:00+00:00,karachi,70,1.0,72.61,12,2,6,69,8,0.08,44.84,90.09,21.01,1002.5,0.42,30.0,14.7,64.0
8422,2026-08-12 07:00:00+00:00,karachi,70,0.0,72.30,12,2,7,70,8,0.07,43.56,90.33,20.94,1001.9,0.42,29.8,15.2,63.0
8423,2026-08-12 08:00:00+00:00,karachi,69,-1.0,72.40,12,2,8,67,8,0.07,42.40,89.29,20.66,1001.7,0.42,30.6,14.9,63.0


In [8]:
print(df.shape)

df.info()

(8424, 19)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8424 entries, 0 to 8423
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   timestamp        8424 non-null   datetime64[ns, UTC]
 1   city             8424 non-null   object             
 2   aqi              8424 non-null   int64              
 3   aqi_change_rate  8424 non-null   float64            
 4   co               8424 non-null   float64            
 5   day              8424 non-null   int64              
 6   day_of_week      8424 non-null   int64              
 7   hour             8424 non-null   int64              
 8   humidity         8424 non-null   int64              
 9   month            8424 non-null   int64              
 10  no2              8424 non-null   float64            
 11  o3               8424 non-null   float64            
 12  pm10             8424 non-null   float64            
 13  pm25   

In [9]:
FEATURES = [
    "hour",
    "day",
    "month",
    "day_of_week",
    "pm25",
    "pm10",
    "o3",
    "no2",
    "so2",
    "co",
    "temperature",
    "humidity",
    "pressure",
    "wind_speed",
]

In [10]:
X = df[FEATURES]

y = df["target_aqi"]

print(X.shape)
print(y.shape)

(8424, 14)
(8424,)


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False,
)

In [12]:
print("Training")

print(X_train.shape)
print(y_train.shape)

print()

print("Testing")

print(X_test.shape)
print(y_test.shape)

Training
(6739, 14)
(6739,)

Testing
(1685, 14)
(1685,)


In [13]:
from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

import numpy as np

In [14]:
model = LinearRegression()

model.fit(X_train, y_train)

print("Linear Regression model trained successfully.")

Linear Regression model trained successfully.


In [15]:
y_pred = model.predict(X_test)

print(y_pred[:10])

[48.44727229 48.66667765 50.16265964 48.9792915  46.5353245  44.03524561
 41.97865748 41.08828317 39.54767241 38.89452346]


In [16]:
mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.4f}")

MAE  : 22.67
RMSE : 28.15
R²   : -1.4510


In [17]:
results = X_test.copy()

results["Actual AQI"] = y_test.values

results["Predicted AQI"] = y_pred

results.head(20)

,hour,day,month,day_of_week,pm25,pm10,o3,no2,so2,co,temperature,humidity,pressure,wind_speed,Actual AQI,Predicted AQI
6739,4,2,6,1,16.06,75.33,38.25,0.08,0.42,76.17,31.2,70,1006.3,13.8,79.0,48.447272
6740,5,2,6,1,15.96,75.20,38.37,0.07,0.41,76.02,32.2,65,1006.3,12.6,79.0,48.666678
6741,6,2,6,1,16.01,76.03,38.02,0.06,0.40,75.86,33.0,59,1006.9,9.1,79.0,50.162660
6742,7,2,6,1,16.15,77.36,37.34,0.05,0.39,75.70,33.5,56,1006.3,8.6,79.0,48.979291
6743,8,2,6,1,16.20,77.85,36.23,0.05,0.38,75.24,34.2,55,1005.4,9.3,79.0,46.535324
6744,9,2,6,1,16.16,77.84,35.16,0.05,0.36,74.90,33.9,58,1004.5,12.2,79.0,44.035246
6745,10,2,6,1,16.15,78.00,34.45,0.05,0.35,74.64,33.1,60,1003.5,12.8,79.0,41.978657
6746,11,2,6,1,16.03,77.57,34.02,0.05,0.33,74.39,32.7,61,1003.1,11.2,80.0,41.088283
6747,12,2,6,1,15.85,76.81,33.91,0.06,0.32,74.22,31.9,67,1002.6,14.0,80.0,39.547672
6748,13,2,6,1,15.74,75.98,33.80,0.07,0.31,74.38,30.7,75,1002.6,15.1,81.0,38.894523


In [18]:
importance = pd.DataFrame({
    "Feature": FEATURES,
    "Coefficient": model.coef_
})

importance = importance.sort_values(
    by="Coefficient",
    key=abs,
    ascending=False,
)

importance

,Feature,Coefficient
2,month,2.248637
8,so2,-2.049001
12,pressure,1.882158
10,temperature,-0.583440
6,o3,0.484045
7,no2,0.394119
1,day,0.369510
4,pm25,0.219630
11,humidity,-0.152758
3,day_of_week,0.058600


In [20]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(model, "../models/linear_regression.pkl")

print("Model saved successfully.")

Model saved successfully.


Linear Regression

In [21]:
from sklearn.ensemble import RandomForestRegressor

In [22]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1,
)

rf_model.fit(X_train, y_train)

print("Random Forest trained successfully.")

Random Forest trained successfully.


In [23]:
rf_predictions = rf_model.predict(X_test)

rf_predictions[:10]

array([79.17060178, 78.95774014, 78.96345212, 78.8904395 , 78.48189575,
       78.33779929, 79.15494598, 79.57319558, 79.56221829, 80.0553734 ])

In [24]:
rf_mae = mean_absolute_error(y_test, rf_predictions)

rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))

rf_r2 = r2_score(y_test, rf_predictions)

print(f"MAE  : {rf_mae:.2f}")
print(f"RMSE : {rf_rmse:.2f}")
print(f"R²   : {rf_r2:.4f}")

MAE  : 14.36
RMSE : 18.95
R²   : -0.1109


In [25]:
rf_results = pd.DataFrame({
    "Actual AQI": y_test.values,
    "Predicted AQI": rf_predictions
})

rf_results.head(20)

,Actual AQI,Predicted AQI
0,79.0,79.170602
1,79.0,78.957740
2,79.0,78.963452
3,79.0,78.890439
4,79.0,78.481896
5,79.0,78.337799
6,79.0,79.154946
7,80.0,79.573196
8,80.0,79.562218
9,81.0,80.055373


In [26]:
importance = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": rf_model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

importance

,Feature,Importance
6,o3,0.372406
1,day,0.149461
2,month,0.148510
9,co,0.100778
4,pm25,0.056335
5,pm10,0.041997
3,day_of_week,0.037075
8,so2,0.031340
12,pressure,0.025291
7,no2,0.011121


In [27]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(
    rf_model,
    "../models/random_forest.pkl"
)

print("Random Forest model saved successfully.")

Random Forest model saved successfully.


In [28]:
comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest"
    ],
    "MAE": [
        mae,
        rf_mae
    ],
    "RMSE": [
        rmse,
        rf_rmse
    ],
    "R²": [
        r2,
        rf_r2
    ]
})

comparison

,Model,MAE,RMSE,R²
0,Linear Regression,22.665750,28.151877,-1.451035
1,Random Forest,14.360593,18.952529,-0.110885
